In [ ]:
import os, torch, torch.nn as nn, torch.optim as optim
from torchvision import models
from data_preprocessing import get_data_loaders
from evaluation_metrics import evaluate_model_metrics, measure_inference_metrics,measure_model_size_and_flops

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
data_dir = "tiny-imagenet-200"
train_loader, val_loader = get_data_loaders(data_dir, batch_size=64, num_workers=4, image_size=224)
print("len(train_loader):", len(train_loader))
print("len(val_loader):", len(val_loader))

In [ ]:
def build_baseline_model(num_classes=200):
    model = models.mobilenet_v2(pretrained=True)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)
    return model

model_baseline = build_baseline_model()
model_baseline = model_baseline.to(device)
print("Baseline MobileNetV2 model built.")

In [ ]:
def train_model(model, train_loader, num_epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
    return model

print("Starting training baseline model...")
model_baseline = train_model(model_baseline, train_loader, num_epochs=10, lr=0.001)
print("Training complete.")

In [ ]:
def evaluate_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    accuracy = 100.0 * correct / total
    return accuracy

accuracy_baseline = evaluate_model(model_baseline, val_loader)
print(f"Baseline Model Validation Accuracy: {accuracy_baseline:.2f}%")

In [ ]:
#%% Define a Differentiable Masked Activation Function
class MaskedActivation(nn.Module):
    def __init__(self, activation=nn.ReLU6(), init_value=1.0):
        super(MaskedActivation, self).__init__()
        self.activation = activation
        # Learnable mask parameter initialized to init_value.
        self.mask = nn.Parameter(torch.tensor(init_value))
    
    def forward(self, x):
        # Use sigmoid to create a soft mask in (0,1)
        mask_value = torch.sigmoid(self.mask)
        # Blend between activation and identity using the soft mask.
        return mask_value * self.activation(x) + (1.0 - mask_value) * x

#%% Replace Activations in the Model
def replace_activations(module):
    """
    Recursively replace all nn.ReLU6 instances with MaskedActivation.
    """
    for name, child in module.named_children():
        if isinstance(child, nn.ReLU6):
            setattr(module, name, MaskedActivation())
        else:
            replace_activations(child)

# Replace activations in baseline model
replace_activations(model_baseline)
print("Replaced nn.ReLU6 with MaskedActivation in the model.")
print(model_baseline) 

In [ ]:
#%% Search Phase: Update Activation Masks
def search_activation_masks(model, train_loader, num_epochs=3, lr=0.0005):
    """
    Fine-tune only the mask parameters for a few epochs to learn their importance.
    Tracking points (print statements) are included to monitor progress.
    """
    # TRACKING: Freeze all parameters except the mask parameters.
    print("Tracking: Freezing all model parameters except for the mask parameters...")
    for param in model.parameters():
        param.requires_grad = False
    
    mask_params = []
    # TRACKING: Identify mask parameters to optimize.
    print("Tracking: Identifying mask parameters to optimize...")
    for module in model.modules():
        if isinstance(module, MaskedActivation):
            module.mask.requires_grad = True
            mask_params.append(module.mask)
    
    print(f"Tracking: Found {len(mask_params)} mask parameters to optimize.")
    
    optimizer = optim.Adam(mask_params, lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    print("Tracking: Starting mask search training...")
    
    # TRACKING: Begin training epochs.
    for epoch in range(num_epochs):
        running_loss = 0.0
        batch_count = 0
        print(f"Tracking: Starting epoch {epoch+1}/{num_epochs}...")
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            batch_count += 1
            # TRACKING: Log batch loss occasionally.
            if batch_count % 50 == 0:
                print(f"  Tracking: Epoch {epoch+1} Batch {batch_count}, Current Loss: {loss.item():.4f}")
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Tracking: Completed epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
    
    # TRACKING: Unfreeze all parameters for further training.
    print("Tracking: Unfreezing all model parameters after mask search training...")
    for param in model.parameters():
        param.requires_grad = True
    
    print("Tracking: Mask search training complete.")
    return model


print("Starting activation mask search...")
model_baseline = search_activation_masks(model_baseline, train_loader, num_epochs=3, lr=0.0005)
print("Activation mask search complete.")

In [ ]:
#%% Prune Redundant Activations
def prune_activations(module, threshold=0.5):
    """
    Replace MaskedActivation modules with either their activation or identity
    based on the learned mask value.
    """
    for name, child in module.named_children():
        if isinstance(child, MaskedActivation):
            # Use the soft mask value to decide: if sigmoid(mask) <= threshold, drop activation.
            mask_value = torch.sigmoid(child.mask).item()
            if mask_value <= threshold:
                setattr(module, name, nn.Identity())
                print(f"Pruned activation at {name} (mask value: {mask_value:.4f})")
            else:
                setattr(module, name, child.activation)
                print(f"Kept activation at {name} (mask value: {mask_value:.4f})")
        else:
            prune_activations(child, threshold=threshold)

prune_activations(model_baseline, threshold=0.5)
print("Pruned redundant activations based on learned masks.")

In [ ]:
#%% Define Convolution Merging Function
def merge_convolutions(conv1, conv2):
    """
    Merge two consecutive convolution layers (both nn.Conv2d) into one.
    Assumes the stride, padding, and kernel size of conv1 are used.
    """
    if not (isinstance(conv1, nn.Conv2d) and isinstance(conv2, nn.Conv2d)):
        raise ValueError("Both layers must be instances of nn.Conv2d.")
    
    print(f"Merging:\n Conv1: {conv1}\n Conv2: {conv2}")
    with torch.no_grad():
        # Reshape and merge weights from conv1 and conv2.
        merged_weight = torch.matmul(
            conv2.weight.view(conv2.out_channels, -1),
            conv1.weight.view(conv1.in_channels, -1)
        )
        merged_weight = merged_weight.view(conv2.out_channels, *conv1.weight.shape[1:])
        # Merge biases (simplified)
        merged_bias = conv2.bias
        if conv1.bias is not None:
            merged_bias = merged_bias + conv2.weight.view(conv2.out_channels, -1).matmul(conv1.bias.view(-1,1)).squeeze()
    
    merged_conv = nn.Conv2d(
        in_channels=conv1.in_channels,
        out_channels=conv2.out_channels,
        kernel_size=conv1.kernel_size,
        stride=conv1.stride,
        padding=conv1.padding
    )
    merged_conv.weight.data = merged_weight
    merged_conv.bias.data = merged_bias
    return merged_conv


In [ ]:
#%% Apply Convolution Merging to the Model
def apply_depthshrinker_merge(model):
    """
    Traverse the model and merge consecutive convolution layers within Sequential containers.
    This function demonstrates a simplified merging process.
    """
    for module_name, module in model.named_modules():
        if isinstance(module, nn.Sequential):
            prev_conv = None
            prev_name = None
            for child_name, child in module.named_children():
                if isinstance(child, nn.Conv2d):
                    if prev_conv is not None:
                        print(f"Merging in {module_name}: {prev_name} and {child_name}")
                        merged_conv = merge_convolutions(prev_conv, child)
                        setattr(module, prev_name, merged_conv)
                        # Replace the second convolution with Identity to remove redundancy.
                        setattr(module, child_name, nn.Identity())
                        # Reset previous convolution reference.
                        prev_conv = None
                    else:
                        prev_conv = child
                        prev_name = child_name
                else:
                    prev_conv = None
    return model

print("Applying convolution merging...")
model_shrunk = apply_depthshrinker_merge(model_baseline)
print("Convolution merging applied.")

In [ ]:
#%% Fine-Tune Model with Tracking
def fine_tune_model(model, train_loader, num_epochs=5, lr=0.001):
    """
    Fine-tune the model and track training progress.
    Tracking points include:
      - Epoch start/end messages.
      - Batch-level logging every 50 batches.
      - Epoch-level average loss.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        batch_count = 0
        print(f"Starting Fine-tune Epoch {epoch+1}/{num_epochs}...")
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            batch_count += 1
            if batch_count % 50 == 0:
                print(f"  Epoch {epoch+1} Batch {batch_count}: Loss = {loss.item():.4f}")
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Completed Fine-tune Epoch {epoch+1}/{num_epochs}, Average Loss: {epoch_loss:.4f}")
    return model

print("Fine-tuning the compressed model...")
model_shrunk = fine_tune_model(model_shrunk, train_loader, num_epochs=5, lr=0.001)
print("Fine-tuning complete.")

In [ ]:
acc = evaluate_model(model_shrunk, val_loader, device)
lat, thr, pwr, eng, edp = measure_inference_metrics(model_shrunk, val_loader, device)
flops, params = measure_model_size_and_flops(model_shrunk)

print(f"\nDepthShrinker Accuracy:  {acc:.2f}%")
print(f"Latency:               {lat*1e3:.2f} ms/img")
print(f"Throughput:            {thr:.2f} imgs/s")
if pwr is not None:
    print(f"Avg GPU Power:          {pwr:.2f} W")
    print(f"Energy per img:         {eng:.4f} J")
    print(f"Energy‑Delay Product:   {edp:.6f} J·s")
print(f"Params:                {params/1e6:.2f} M")
if flops is not None:
    print(f"FLOPs:                 {flops:.2f} GFLOPs")

torch.save(model_shrunk.state_dict(), "weights/mobilenetv2_depthshrinker.pth")
print("✓ weights/mobilenetv2_depthshrinker.pth saved")